# OpenClaw 架构概览 -- 微核 + 插件 + 网关设计

## 什么是 OpenClaw?

OpenClaw 是 **B站（Bilibili）开源的 AI Agent 框架**，专为商业化场景设计。
它采用 **微核架构（Microkernel Architecture）**，将核心调度逻辑与业务功能解耦，
通过插件系统实现灵活扩展。

### 核心组件

| 组件 | 职责 | 类比 |
|------|------|------|
| **Gateway（网关）** | API 入口、路由分发、鉴权限流 | 大门保安 |
| **Agent（智能体）** | 状态管理、意图理解、决策编排 | 大脑 |
| **Runner（运行器）** | 执行引擎、工具调用、结果聚合 | 手脚 |
| **Plugin（插件）** | 业务能力封装（广告查询、推荐等） | 技能包 |
| **Loop（循环控制）** | ReAct/Plan-Execute 等执行循环 | 思考节奏 |

在B站商业化中，OpenClaw 支撑了广告投放助手、商业智能问答、创作者运营 Agent 等场景。

In [ ]:
# OpenClaw 架构 ASCII 示意图

architecture = """
╔══════════════════════════════════════════════════════════════╗
║                   OpenClaw 架构总览                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║   [用户请求]                                                 ║
║       │                                                      ║
║       ▼                                                      ║
║   ┌──────────┐   鉴权/限流/路由                              ║
║   │ Gateway   │──────────────────┐                           ║
║   └──────────┘                   │                           ║
║       │                          ▼                           ║
║       ▼                   ┌─────────────┐                    ║
║   ┌──────────┐            │  中间件链     │                   ║
║   │  Agent   │◄───────────┤ (Auth/Log)  │                   ║
║   │ (决策中心)│            └─────────────┘                    ║
║   └──────────┘                                               ║
║       │                                                      ║
║       ▼  选择执行策略                                        ║
║   ┌──────────┐                                               ║
║   │  Runner  │  ◄── Loop (ReAct / Plan-Execute)             ║
║   │ (执行引擎)│                                              ║
║   └──────────┘                                               ║
║       │                                                      ║
║       ▼  调用插件                                            ║
║   ┌──────────┬──────────┬──────────┐                        ║
║   │ Plugin A │ Plugin B │ Plugin C │  ...                   ║
║   │ (广告查询)│ (推荐)   │ (报表)   │                        ║
║   └──────────┴──────────┴──────────┘                        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(architecture)

In [ ]:
# 模拟一个请求在 OpenClaw 中的流转过程

import time

class OpenClawRequestFlow:
    """演示请求流转：Gateway -> Agent -> Runner -> Plugin -> 返回"""

    def __init__(self):
        self.trace = []  # 记录请求经过的组件

    def gateway(self, request: dict) -> dict:
        """网关层：鉴权、限流、路由"""
        self.trace.append("1. Gateway: 接收请求, 鉴权通过, 路由到 Agent")
        # 附加鉴权信息
        request["auth"] = {"user_id": "biz_001", "role": "advertiser"}
        return self.agent(request)

    def agent(self, request: dict) -> dict:
        """Agent层：意图理解、决策"""
        self.trace.append("2. Agent: 解析意图='查询广告效果', 决定调用 ad_query 插件")
        plan = {"plugin": "ad_query", "params": request["query"]}
        return self.runner(plan)

    def runner(self, plan: dict) -> dict:
        """Runner层：执行计划、调用插件"""
        self.trace.append(f"3. Runner: 执行计划, 调用插件 [{plan['plugin']}]")
        result = self.plugin_execute(plan["plugin"], plan["params"])
        self.trace.append("5. Runner: 聚合结果, 返回给 Agent")
        return {"status": "success", "data": result}

    def plugin_execute(self, plugin_name: str, params: str) -> str:
        """插件执行"""
        self.trace.append(f"4. Plugin[{plugin_name}]: 查询数据库, 返回广告数据")
        return f"广告计划'双11大促'的 CTR=3.2%, CPM=¥15.8"


# 模拟 B站广告主查询广告效果
flow = OpenClawRequestFlow()
request = {"query": "查询我的双11广告投放效果", "channel": "web"}
response = flow.gateway(request)

print("=== 请求流转轨迹 ===")
for step in flow.trace:
    print(f"  {step}")
print(f"\n=== 最终响应 ===")
print(f"  状态: {response['status']}")
print(f"  数据: {response['data']}")

## 核心设计理念：微核架构 vs 单体架构

| 维度 | 微核架构（OpenClaw） | 单体架构 |
|------|---------------------|----------|
| **核心** | 只保留调度/路由/生命周期管理 | 所有逻辑在一个代码库 |
| **扩展性** | 新增插件即可扩展能力 | 修改主代码、重新部署 |
| **隔离性** | 插件故障不影响核心 | 一个模块崩溃可能影响全局 |
| **部署** | 插件可独立部署/热更新 | 整体打包部署 |
| **适用场景** | B站多业务线并行开发 | 早期小规模项目 |

### 为什么B站选择微核？

B站商业化涉及 **广告、电商、直播、会员** 等多条业务线，每条线都需要定制 Agent 能力。
微核架构让各团队可以 **独立开发插件**，通过统一的 Gateway 对外暴露服务，
Runner 负责编排执行，避免了单体架构下的代码耦合和部署瓶颈。

## 面试速记 & 高频考点

### Q1: 请描述 OpenClaw 的整体架构设计
**答题思路**: Gateway(入口) -> Agent(决策) -> Runner(执行) -> Plugin(能力) -> Loop(控制)，
强调微核设计，核心只做调度，业务逻辑下沉到插件。

### Q2: 微核架构相比单体架构有什么优势？
**答题思路**: 扩展性（插件热插拔）、隔离性（故障不传播）、
团队协作（各业务线独立开发插件）。结合B站多业务线的实际场景作答。

### Q3: Gateway 在 Agent 框架中的作用是什么？
**答题思路**: 统一入口、协议转换、鉴权限流、路由分发。
类比 API Gateway 在微服务架构中的角色。

### Q4: Loop 组件支持哪些执行模式？
**答题思路**: ReAct（推理-行动交替）、Plan-Execute（先规划后执行）、
以及自定义循环策略。不同场景选择不同 Loop，如简单查询用单步，复杂分析用 Plan-Execute。

> **面试加分项**: 能画出架构图并解释数据流向，说明各组件的职责边界。